In [43]:
import os
import json
import pandas as pd
from dotenv import load_dotenv

#load .env
load_dotenv()

with open("config/config.json","r") as f:
    config = json.load(f)

print(config)

{'llm': {'openai_model': 'gpt-4o-mini', 'gemini_model': 'gemini-1.5-flash', 'temperature': 0.3}, 'evaluation': {'criteria': ['tone_empathy', 'knowledge_accuracy', 'resolution_quality'], 'score_range': [1, 5]}, 'classification': {'labels': ['billing', 'claims', 'complaint', 'general_query']}}


In [44]:
df = pd.read_csv("data/transcripts.csv")

print("Data Loaded")
print(df.head())

Data Loaded
   call_id agent_name                                         transcript  \
0        1      Alice  Customer: I was charged twice for my premium t...   
1        2        Bob  Customer: My claim has been pending for 2 week...   
2        3    Charlie  Customer: This is the third time I'm calling! ...   
3        4      Diana  Customer: Can you explain my coverage details?...   

  expected_call_type  
0            billing  
1             claims  
2          complaint  
3      general_query  


In [45]:
df.tail(2)

,call_id,agent_name,transcript,expected_call_type
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint
3,4,Diana,Customer: Can you explain my coverage details?...,general_query


In [46]:
df.head()

,call_id,agent_name,transcript,expected_call_type
0,1,Alice,Customer: I was charged twice for my premium t...,billing
1,2,Bob,Customer: My claim has been pending for 2 week...,claims
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint
3,4,Diana,Customer: Can you explain my coverage details?...,general_query


In [47]:
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

def load_llm(llm):
    provider = os.getenv("LLM_PROVIDER")

    if provider == "openai":
        llm = ChatOpenAI(model = config["llm"].get("openai_model", "gpt-4o-mini"), temperature = config["llm"].get("temperature", 0.3) )
    elif provider == "gemini":
        llm = ChatGoogleGenerativeAI(model = config["llm"].get("gemini_model", "gemini-1.5-flash"), temperature = config["llm"].get("temperature", 0.3) )
    else:
        raise ValueError("LLM_PROVIDER")
    return llm

llm = load_llm(config)

print(f"llm initialized using provider: {os.getenv('LLM_PROVIDER')}")

llm initialized using provider: openai


In [48]:
response = llm.invoke("Say 'setup successful' in one short sentence")

print(response)

content='Setup successful.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 3, 'prompt_tokens': 16, 'total_tokens': 19, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_19227531c6', 'id': 'chatcmpl-EVFSVYxvJaIGqrxxEDnkfBfg6LnRP', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a106d9-c93c-7263-8bc2-40ff29d34f97-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 16, 'output_tokens': 3, 'total_tokens': 19, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [49]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


class ClassificationOutput(BaseModel):
    call_type: str = Field(description = "Type of customer call")
    confidence: float = Field(description = "Confidence score between 0 and 1")

parser = PydanticOutputParser(pydantic_object = ClassificationOutput)


#prompt template
prompt = PromptTemplate(
    template = """
    You are called classification assistant. 

    classify the following customer support transcript into one of the categories:
    {labels}

    Transcript:
    {transcript}

    {format_instructions}   
    """, 
    input_variables = ["transcript"], 
    partial_variables = {
        "format_instructions": parser.get_format_instructions(), 
            "labels": config["classification"]["labels"] 
                  }
)

#chain

classification_chain = prompt | llm | parser

#test chain with one transcript

# df.iloc[0]["transcript"]

response = classification_chain.invoke(df.iloc[0]["transcript"])

print(f"Classification result: {response}")

Classification result: call_type='billing' confidence=0.95


In [50]:
from tqdm import tqdm

results = []

for i , row in tqdm(df.iterrows(), total = len(df), desc = "Classifying Calls"):
    classification_chain = prompt | llm | parser

    try:
        response = classification_chain.invoke(row["transcript"])
        results.append({
            "call_id": row["call_id"],
            "transcript": row["transcript"],
            "predicted_call_type": response.call_type,
            "confidence": response.confidence
        })

    except Exception as e:
        print(f"Error classifying call {row['call_id']}: {e}")
        results.append({
            "call_id": row["call_id"],
            "transcript": row["transcript"],
            "predicted_call_type": None,
            "confidence": None
        })

results_df = pd.DataFrame(results)
df = df.merge(results_df, on = "call_id")

print(df)

Classifying Calls: 100%|██████████| 4/4 [00:03<00:00,  1.01it/s]

   call_id agent_name                                       transcript_x  \
0        1      Alice  Customer: I was charged twice for my premium t...   
1        2        Bob  Customer: My claim has been pending for 2 week...   
2        3    Charlie  Customer: This is the third time I'm calling! ...   
3        4      Diana  Customer: Can you explain my coverage details?...   

  expected_call_type                                       transcript_y  \
0            billing  Customer: I was charged twice for my premium t...   
1             claims  Customer: My claim has been pending for 2 week...   
2          complaint  Customer: This is the third time I'm calling! ...   
3      general_query  Customer: Can you explain my coverage details?...   

  predicted_call_type  confidence  
0             billing        0.95  
1              claims        0.95  
2           complaint        0.90  
3       general_query        0.95  


In [51]:
df

,call_id,agent_name,transcript_x,expected_call_type,transcript_y,predicted_call_type,confidence
0,1,Alice,Customer: I was charged twice for my premium t...,billing,Customer: I was charged twice for my premium t...,billing,0.95
1,2,Bob,Customer: My claim has been pending for 2 week...,claims,Customer: My claim has been pending for 2 week...,claims,0.95
2,3,Charlie,Customer: This is the third time I'm calling! ...,complaint,Customer: This is the third time I'm calling! ...,complaint,0.90
3,4,Diana,Customer: Can you explain my coverage details?...,general_query,Customer: Can you explain my coverage details?...,general_query,0.95


In [52]:
def route_call(call_type):
    if call_type == "billing":
        return ["knowledge_accuracy", "resolution quality"]
    elif call_type == "claims":
        return ["knowledge_accuracy", "resolution quality"]
    elif call_type == "complaint":
        return ["tone_empathy", "resolution quality"]
    elif call_type == "general_query":
        return ["knowledge_accuracy"] 
    else:
        return ["knowledge_accuracy"] 

df["evaluation_plan"] = df["predicted_call_type"].apply(route_call)

print("Routing Applied\n")
print(df[["call_id", "predicted_call_type", "evaluation_plan"]])

Routing Applied

   call_id predicted_call_type                           evaluation_plan
0        1             billing  [knowledge_accuracy, resolution quality]
1        2              claims  [knowledge_accuracy, resolution quality]
2        3           complaint        [tone_empathy, resolution quality]
3        4       general_query                      [knowledge_accuracy]


In [53]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

min_score, max_score = config["evaluation"]["score_range"]


class ToneEvaluationOutput(BaseModel):
    tone_empathy_score: int = Field(description = f"Tone and empathy score between {min_score} and {max_score}", ge = min_score, le = max_score)
    justification: str = Field(description = "Short reason for the score")

tone_parser = PydanticOutputParser(pydantic_object = ToneEvaluationOutput)


#prompt template
tone_prompt = PromptTemplate(
    template = """
    You are a quality assurance assistant for a customer support team.

    Evaluate the agent's tone and empathy in the following customer support transcript.
    Check whether the agent was polite, acknowledged the customer's feelings, and stayed calm and respectful.

    Score it from {min_score} (poor) to {max_score} (excellent).

    Transcript:
    {transcript}

    {format_instructions}
    """,
    input_variables = ["transcript"],
    partial_variables = {
        "format_instructions": tone_parser.get_format_instructions(),
            "min_score": min_score,
            "max_score": max_score
                  }
)

#chain

tone_chain = tone_prompt | llm | tone_parser

#test chain with one transcript

tone_response = tone_chain.invoke(df.iloc[0]["transcript"])

print(f"Tone evaluation result: {tone_response}")


KeyError: 'transcript'

In [54]:
from tqdm import tqdm

tone_results = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating Tone"):
    # only evaluate calls whose evaluation plan includes tone_empathy
    if "tone_empathy" not in row["evaluation_plan"]:
        continue

    try:
        response = tone_chain.invoke(row["transcript"])
        tone_results.append({
            "call_id": row["call_id"],
            "tone_empathy_score": response.tone_empathy_score,
            "tone_justification": response.justification,
        })

    except Exception as e:
        print(f"Error evaluating tone for call {row['call_id']}: {e}")
        tone_results.append({
            "call_id": row["call_id"],
            "tone_empathy_score": None,
            "tone_justification": None,
        })

tone_df = pd.DataFrame(tone_results, columns=["call_id", "tone_empathy_score", "tone_justification"])
df = df.merge(tone_df, on="call_id", how="left")

print("Tone Evaluation Applied\n")
print(df[["call_id", "predicted_call_type", "tone_empathy_score", "tone_justification"]])


Evaluating Tone: 100%|██████████| 4/4 [00:00<00:00, 1977.51it/s]

Error evaluating tone for call 3: 'transcript'
Tone Evaluation Applied

   call_id predicted_call_type tone_empathy_score tone_justification
0        1             billing                NaN                NaN
1        2              claims                NaN                NaN
2        3           complaint               None               None
3        4       general_query                NaN                NaN
